# 04 — Full Pipeline (Colab / GPU)

Run cells top-to-bottom. The pipeline resumes automatically if checkpoints or Phi already exist.

**Resume order:**
1. `influence_matrix.npy` found locally or on Hub → skip surrogate training + influence computation
2. `checkpoints/` found locally or on Hub → skip surrogate training, recompute Phi
3. Nothing found → full run

In [ ]:
# ── Cell 1: RunPod setup (run once per session) ──
# Uncomment and run this block when starting a fresh RunPod session.

# !git clone https://github.com/flakoash/influece_driven_curriculum_sorter.git /workspace/babylm
# %cd /workspace/babylm
# !pip install -e ".[dev]" --quiet   # installs core deps + huggingface_hub

# Download BabyLM dataset (skips files already present)
# !mkdir -p /workspace/datasets
# !hf download BabyLM-community/BabyLM-2026-Strict-Small \
#   --repo-type dataset --local-dir /workspace/datasets/BabyLM-2026-Strict-Small

print("Setup complete.")

In [ ]:
# ── Cell 2: Config — edit these before running ──
RUN_ID = "05"

MODEL        = "BabyLM-community/BabyLM-2026-Baseline-GPT2-Strict-Small"
DATA_DIR     = "/workspace/datasets/BabyLM-2026-Strict-Small"
OUTPUT_DIR   = f"/workspace/outputs/run_{RUN_ID}"                  # persists Phi + curriculum

SAMPLE_FRAC  = 1.0     # 1.0 = full corpus; 0.05 = 5% (~36k docs) for fast iteration
SURROGATE_EPOCHS = 2   # T checkpoints produced
MAX_SEQ_LEN  = 128
STRATEGY     = "epoch_wise"   # "epoch_wise" (C~) or "cumulative" (C^E)
N_GROUPS     = 2              # only used when STRATEGY="cumulative"
SEED         = 0

# ── Smoke test: set True to run only SMOKE_N_DOCS docs ──────────────────────
# Caps corpus to 300 docs so the full pipeline completes in < 2 min.
# Use this to catch OOM errors fast before committing to a full run.
SMOKE_TEST    = True
SMOKE_N_DOCS  = 300

# ── Influence computation memory knobs ────────────────────────────────────────
# pass1_batch_size: docs per mini-batch backward (mean gradient)
#   fp16 pass1 memory: B×128×16384×2B = 0.5 GB at B=256 (3090 safe)
#   fp32 pass1 memory: B×128×16384×4B = 1.0 GB at B=128 (T4 safe at B=64)
# pass2_batch_size: docs per vmap(jvp) batch (3090: 64; T4: 8)
PASS1_BATCH_SIZE = 256   # 3090 fp16 safe; reduce to 64 on T4
PASS2_BATCH_SIZE = 64    # 3090: 64 (~1.3 GB peak); T4: 8 (~315 MB peak)

# HF Hub repos — set to None to skip Hub push/pull
HUB_MODEL_REPO   = f"flakoash/babylm-surrogate-run{RUN_ID}"
HUB_DATASET_REPO = f"flakoash/babylm-curriculum-run{RUN_ID}"
HF_TOKEN         = None   # or set via: import os; HF_TOKEN = os.environ["HF_TOKEN"]

print(f"Strategy    : {STRATEGY}" + (f" n_groups={N_GROUPS}" if STRATEGY == "cumulative" else ""))
print(f"SMOKE_TEST  : {SMOKE_TEST}" + (f"  ({SMOKE_N_DOCS} docs)" if SMOKE_TEST else ""))
print(f"Pass1 batch : {PASS1_BATCH_SIZE}  |  Pass2 batch: {PASS2_BATCH_SIZE}")
print(f"Output      : {OUTPUT_DIR}")

In [ ]:
# ── Cell 3: Imports + device ──
import json, os, re, random, sys
from collections import Counter
from pathlib import Path

# Reduce CUDA memory fragmentation — set before importing torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GPT2Config, GPT2LMHeadModel

# Allow running without pip install -e . (e.g. fresh RunPod with just git clone)
for _candidate in [Path("../src"), Path("src"), Path("/workspace/babylm/src")]:
    if _candidate.exists() and str(_candidate.resolve()) not in sys.path:
        sys.path.insert(0, str(_candidate.resolve()))

from influence_curriculum.data import DataConfig, load_documents
from influence_curriculum.train import TrainingConfig, train_surrogate
from influence_curriculum.score import InfluenceConfig, compute_influence_matrix
from influence_curriculum.curriculum import CurriculumConfig, build_curriculum

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT / "checkpoints"

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
# ── Cell 4: BabyLM segmentation rules + load documents ──
def segment_childes(text):
    docs = []
    for line in text.splitlines():
        if re.match(r'^\[', line.strip()):
            continue
        line = re.sub(r'^\*[A-Z]+:\t', '', line)
        line = re.sub(r'\[.*?\]', '', line).strip()
        if line:
            docs.append(line)
    return docs

def segment_simple_wiki(text):
    docs, current = [], []
    for line in text.splitlines():
        if re.match(r'^= = =', line.strip()):
            if current:
                docs.append(' '.join(current))
            title = re.sub(r'= = =', '', line).strip()
            current = [title] if title else []
        elif line.strip():
            current.append(line.strip())
    if current:
        docs.append(' '.join(current))
    return docs

data_cfg = DataConfig(
    doc_boundary={
        "childes.train.txt": segment_childes,
        "simple_wiki.train.txt": segment_simple_wiki,
    },
    default_boundary="line",
    min_doc_tokens=3,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

texts_all, ids_all = load_documents(DATA_DIR, data_cfg)
print(f"Full corpus: {len(texts_all):,} docs")

# Optional subsample
if SAMPLE_FRAC < 1.0:
    rng = random.Random(SEED)
    by_src = {}
    for t, i in zip(texts_all, ids_all):
        by_src.setdefault(i.split("#")[0], []).append((t, i))
    N = round(len(texts_all) * SAMPLE_FRAC)
    sampled = []
    for src, items in sorted(by_src.items()):
        n = max(1, round(N * len(items) / len(texts_all)))
        sampled.extend(rng.sample(items, min(n, len(items))))
    rng.shuffle(sampled)
    sampled = sampled[:N]
    texts_all, ids_all = zip(*sampled)
    texts_all, ids_all = list(texts_all), list(ids_all)

# ── Smoke test: cap to SMOKE_N_DOCS docs for fast OOM detection ──────────────
if SMOKE_TEST:
    texts_all = texts_all[:SMOKE_N_DOCS]
    ids_all   = ids_all[:SMOKE_N_DOCS]
    print(f"SMOKE TEST: capped to {len(texts_all)} docs (set SMOKE_TEST=False for full run)")

texts, doc_ids = texts_all, ids_all
src_counts = Counter(i.split("#")[0] for i in doc_ids)
print(f"Working corpus: {len(texts):,} docs ({100*SAMPLE_FRAC:.0f}% of full)")
for src, n in sorted(src_counts.items()):
    print(f"  {src:30s}: {n:6,} docs")

In [ ]:
# ── Cell 5: Resume check ──
# Determines what needs to run: nothing / just Phi / full train

phi_path  = OUT / "influence_matrix.npy"
ids_path  = OUT / "doc_ids.json"
Phi, checkpoint_paths = None, None

def _valid_ckpt(p: Path) -> bool:
    """A real checkpoint dir: is a directory, not hidden, has config.json."""
    return p.is_dir() and not p.name.startswith(".") and (p / "config.json").exists()

# 1. Try local Phi
if phi_path.exists() and ids_path.exists():
    saved_ids = json.loads(ids_path.read_text())
    if saved_ids == doc_ids:
        Phi = np.load(str(phi_path))
        print(f"Resumed Phi from local disk  {Phi.shape}")
    else:
        print("Local Phi found but doc_ids mismatch — will recompute.")

# 2. Try Hub Phi
if Phi is None and HUB_DATASET_REPO:
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(HUB_DATASET_REPO, "influence_matrix.npy", local_dir=str(OUT),
                        repo_type="dataset", token=HF_TOKEN)
        hf_hub_download(HUB_DATASET_REPO, "doc_ids.json", local_dir=str(OUT),
                        repo_type="dataset", token=HF_TOKEN)
        saved_ids = json.loads(ids_path.read_text())
        if saved_ids == doc_ids:
            Phi = np.load(str(phi_path))
            print(f"Resumed Phi from Hub  {Phi.shape}")
        else:
            print("Hub Phi doc_ids mismatch — will recompute.")
    except Exception as e:
        print(f"Hub Phi not found ({e})")

# 3. Try local checkpoints (to recompute Phi without retraining)
if Phi is None and CKPT_DIR.exists():
    ckpts = sorted(p for p in CKPT_DIR.iterdir() if _valid_ckpt(p))
    if ckpts:
        checkpoint_paths = [str(p) for p in ckpts]
        print(f"Found local checkpoints: {[p.name for p in ckpts]}")

# 4. Try Hub checkpoints
if Phi is None and checkpoint_paths is None and HUB_MODEL_REPO:
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=HUB_MODEL_REPO, local_dir=str(CKPT_DIR),
                          repo_type="model", token=HF_TOKEN)
        ckpts = sorted(p for p in CKPT_DIR.iterdir() if _valid_ckpt(p))
        if ckpts:
            checkpoint_paths = [str(p) for p in ckpts]
            print(f"Downloaded checkpoints from Hub: {[p.name for p in ckpts]}")
    except Exception as e:
        print(f"Hub checkpoints not found ({e}) — will train from scratch.")

if Phi is not None:
    print("→ skipping surrogate training and influence computation")
elif checkpoint_paths:
    print("→ skipping surrogate training, will recompute Phi from checkpoints")
else:
    print("→ full run: train surrogate + compute Phi")

In [ ]:
# ── Cell 6: Surrogate training (skipped if resumed) ──
if Phi is None and checkpoint_paths is None:
    _train_model = GPT2LMHeadModel(GPT2Config(
        n_embd=384,
        n_layer=8,
        n_head=6,
        n_inner=384 * 4,
        vocab_size=16384,
        n_positions=1024,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    ))
    train_cfg = TrainingConfig(
        epochs=SURROGATE_EPOCHS,
        per_device_batch_size=64,
        effective_batch_size=256,
        max_seq_len=MAX_SEQ_LEN,
        lr_scheduler="cosine",
        fp16=(DEVICE == "cuda"),
    )
    print(f"Training surrogate: GPT2 from scratch  ({sum(p.numel() for p in _train_model.parameters()):,} params)")
    checkpoint_paths = train_surrogate(
        _train_model, tokenizer, texts, str(OUT), train_cfg, seed=SEED, device=DEVICE
    )
    print(f"Checkpoints: {[Path(p).name for p in checkpoint_paths]}")

    # Free training model + GPU cache before influence computation
    del _train_model
    import gc; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    if HUB_MODEL_REPO:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(repo_id=HUB_MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
        print(f"Pushing checkpoints → {HUB_MODEL_REPO}")
        api.upload_folder(
            folder_path=str(CKPT_DIR), repo_id=HUB_MODEL_REPO,
            repo_type="model", token=HF_TOKEN, commit_message="surrogate checkpoints",
        )
else:
    print("Surrogate training skipped.")

In [ ]:
# ── Cell 7: Influence matrix (skipped if Phi already loaded) ──
if Phi is None:
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU before influence: {used:.1f} / {total:.1f} GB used")

    print(f"Tokenizing {len(texts):,} docs...")
    batch_out = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_tensors=None,
    )
    encodings = [
        {k: torch.tensor([v[i]]) for k, v in batch_out.items()}
        for i in range(len(texts))
    ]
    del batch_out
    gc.collect()

    print(f"Done. Computing Phi from {len(checkpoint_paths)} checkpoints...")
    infl_cfg = InfluenceConfig(
        pass1_batch_size=PASS1_BATCH_SIZE,
        pass2_batch_size=PASS2_BATCH_SIZE,
        fp16=(DEVICE == "cuda"),
    )
    Phi = compute_influence_matrix(checkpoint_paths, encodings, infl_cfg, device=DEVICE)
    np.save(str(phi_path), Phi)
    ids_path.write_text(json.dumps(doc_ids))
    print(f"Phi saved  {Phi.shape}  range=[{Phi.min():.4f}, {Phi.max():.4f}]")
else:
    print("Influence computation skipped.")

In [ ]:
# ── Cell 8: Build curriculum ──
curr_cfg = CurriculumConfig(
    strategy=STRATEGY,
    n_groups=N_GROUPS,
)
build_curriculum(Phi, texts, doc_ids, curr_cfg, str(OUT), seed=SEED)

curr_dir = OUT / "curriculum"
epoch_files = sorted(curr_dir.glob("epoch_*.jsonl"))

# Write .txt mirrors: one doc per line, blank line between docs
for jf in epoch_files:
    tf = jf.with_suffix(".txt")
    with jf.open() as src, tf.open("w") as dst:
        for line in src:
            dst.write(json.loads(line)["text"] + "\n\n")

print(f"Curriculum written ({STRATEGY}):")
for jf in epoch_files:
    n = sum(1 for _ in jf.open())
    print(f"  {jf.name}: {n:,} docs  →  {jf.with_suffix('.txt').name}")

In [ ]:
# ── Cell 9: Push Phi + curriculum to HF Hub (optional) ──
if HUB_DATASET_REPO:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HUB_DATASET_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
    print(f"Pushing artifacts → {HUB_DATASET_REPO}")
    api.upload_folder(
        folder_path=str(OUT),
        repo_id=HUB_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
        ignore_patterns=["checkpoints/**"],
        commit_message=f"curriculum strategy={STRATEGY} n_groups={N_GROUPS}",
    )
    print("Done.")
else:
    print("HUB_DATASET_REPO not set — skipping Hub push.")

In [ ]:
# ── Cell 10: Metrics ──
import matplotlib.pyplot as plt

sources = sorted(src_counts.keys())
colors  = plt.cm.tab10.colors

# Per-source influence score summary
print("Influence score summary (mean across checkpoints):")
agg = Phi.mean(axis=1)
for src in sources:
    mask = np.array([i.split("#")[0] == src for i in doc_ids])
    s = agg[mask]
    print(f"  {src:30s}: mean={s.mean():.4f}  std={s.std():.4f}  n={mask.sum():,}")

# Score distribution per source
fig, ax = plt.subplots(figsize=(10, 4))
for color, src in zip(colors, sources):
    mask = np.array([i.split("#")[0] == src for i in doc_ids])
    ax.hist(agg[mask], bins=40, alpha=0.55, label=src.replace(".train", ""), color=color)
ax.set_xlabel("mean influence score")
ax.set_ylabel("docs")
ax.set_title(f"Influence distribution by source  ({STRATEGY})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Curriculum epoch sizes bar chart
epoch_files = sorted((OUT / "curriculum").glob("epoch_*.jsonl"))
epoch_sizes = [sum(1 for _ in p.open()) for p in epoch_files]
fig, ax = plt.subplots(figsize=(max(6, len(epoch_files) * 1.5), 4))
ax.bar(range(len(epoch_files)), epoch_sizes, color="steelblue")
ax.set_xticks(range(len(epoch_files)))
ax.set_xticklabels([p.name for p in epoch_files], rotation=30, ha="right")
ax.set_ylabel("docs in epoch")
ax.set_title(f"Curriculum epoch sizes  ({STRATEGY})")
for i, n in enumerate(epoch_sizes):
    ax.text(i, n + len(doc_ids)*0.005, f"{n:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()